<a href="https://colab.research.google.com/github/hemachandranrcs24-spec/Ai-Lab-5H/blob/main/HEMACHANDRANR_1BM24CS115WEEK4WEEK3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import heapq
import copy

class EightPuzzleProblem:
    def __init__(self, start_state, goal_state):
        self.initial_state = start_state
        self.goal_state = goal_state

    def goal_test(self, state):
        return state == self.goal_state

    def find_blank(self, state):
        for r in range(3):
            for c in range(3):
                if state[r][c] == 0:
                    return r, c

    def actions(self, state):
        r, c = self.find_blank(state)
        possible_actions = []
        if r > 0: possible_actions.append("UP")
        if r < 2: possible_actions.append("DOWN")
        if c > 0: possible_actions.append("LEFT")
        if c < 2: possible_actions.append("RIGHT")
        return possible_actions

    def result(self, state, action):
        next_state = copy.deepcopy(state)
        r, c = self.find_blank(state)
        if action == "UP":
            next_state[r][c], next_state[r-1][c] = next_state[r-1][c], next_state[r][c]
        elif action == "DOWN":
            next_state[r][c], next_state[r+1][c] = next_state[r+1][c], next_state[r][c]
        elif action == "LEFT":
            next_state[r][c], next_state[r][c-1] = next_state[r][c-1], next_state[r][c]
        elif action == "RIGHT":
            next_state[r][c], next_state[r][c+1] = next_state[r][c+1], next_state[r][c]
        return next_state

    def step_cost(self, state, action, next_state):
        return 1

    def heuristic(self, state):
        misplaced = 0
        for r in range(3):
            for c in range(3):
                val = state[r][c]
                if val != 0 and val != self.goal_state[r][c]:
                    misplaced += 1
        return misplaced

class Node:
    def __init__(self, state, parent=None, action=None, g_cost=0, h_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g_cost
        self.h = h_cost
        self.f = g_cost + h_cost

    def __lt__(self, other):
        return self.f < other.f

def format_grid(state):
    return "\n".join([" ".join(map(str, row)) for row in state])

def print_output(node):
    path = []
    current = node
    while current is not None:
        path.append(current)
        current = current.parent
    path.reverse()

    directions = []
    for step, n in enumerate(path):
        action_str = n.action if n.action else "START"
        if n.action:
            directions.append(n.action)
        print(f"Step {step}: {action_str}")
        print(format_grid(n.state))
        print(f"g={n.g}, h={n.h}, f={n.f}\n")

    print("Directions:")
    print(" -> ".join(directions) if directions else "None")

def a_star_search(problem):
    start_node = Node(state=problem.initial_state, g_cost=0, h_cost=problem.heuristic(problem.initial_state))
    frontier = []
    heapq.heappush(frontier, start_node)
    explored = set()

    while frontier:
        n = heapq.heappop(frontier)
        state_str = str(n.state)
        if state_str in explored:
            continue
        explored.add(state_str)

        if problem.goal_test(n.state):
            print_output(n)
            return True

        for action in problem.actions(n.state):
            child_state = problem.result(n.state, action)
            if str(child_state) in explored:
                continue
            child_g = n.g + problem.step_cost(n.state, action, child_state)
            child_h = problem.heuristic(child_state)
            child_node = Node(state=child_state, parent=n, action=action, g_cost=child_g, h_cost=child_h)
            heapq.heappush(frontier, child_node)
    return False

def get_board(name):
    print(f"Enter 3x3 matrix for {name}:")
    board = []
    for _ in range(3):
        board.append([int(x) for x in input().strip().split()])
    return board

if __name__ == "__main__":
    initial = get_board("INITIAL")
    goal = get_board("GOAL")
    print()
    a_star_search(EightPuzzleProblem(initial, goal))


Enter 3x3 matrix for INITIAL:
2 8 3
1 6 4
7 0 5 
Enter 3x3 matrix for GOAL:
1 2 3 
8 0 4 
7 6 5 

Step 0: START
2 8 3
1 6 4
7 0 5
g=0, h=4, f=4

Step 1: UP
2 8 3
1 0 4
7 6 5
g=1, h=3, f=4

Step 2: UP
2 0 3
1 8 4
7 6 5
g=2, h=3, f=5

Step 3: LEFT
0 2 3
1 8 4
7 6 5
g=3, h=2, f=5

Step 4: DOWN
1 2 3
0 8 4
7 6 5
g=4, h=1, f=5

Step 5: RIGHT
1 2 3
8 0 4
7 6 5
g=5, h=0, f=5

Directions:
UP -> UP -> LEFT -> DOWN -> RIGHT


In [2]:
import copy

class FourQueensProblem:
    def __init__(self, initial_state):
        self.initial_state = initial_state

    def get_cost(self, state):
        attacking_pairs = 0
        n = len(state)
        for i in range(n):
            for j in range(i + 1, n):
                if state[i] == state[j] or abs(state[i] - state[j]) == abs(i - j):
                    attacking_pairs += 1
        return attacking_pairs

    def get_neighbors(self, state):
        neighbors = []
        n = len(state)
        for i in range(n):
            for j in range(i + 1, n):
                neighbor_state = list(state)
                neighbor_state[i], neighbor_state[j] = neighbor_state[j], neighbor_state[i]
                cost = self.get_cost(neighbor_state)
                neighbors.append({
                    "state": neighbor_state,
                    "cost": cost
                })
        return neighbors

def format_state(state):
    return " ".join(map(str, state))

def hill_climbing(problem):
    current_state = problem.initial_state
    current_cost = problem.get_cost(current_state)
    first_step = True

    print(f"Initial State: {format_state(current_state)}\n")

    # If the initial state is already perfect
    if current_cost == 0:
        print(f"Final State: {format_state(current_state)}")
        return

    while True:
        if first_step:
            print(f"Solution: Neighbours of {format_state(current_state)} (cost is {current_cost}.)\n")
            first_step = False
        else:
            print(f"Neighbours of {format_state(current_state)} (cost is {current_cost}.)\n")

        neighbors = problem.get_neighbors(current_state)

        best_neighbor = None
        min_cost = current_cost

        for nb in neighbors:
            if nb["cost"] < min_cost:
                min_cost = nb["cost"]
                best_neighbor = nb
            elif nb["cost"] == min_cost and best_neighbor is None and nb["cost"] < current_cost:
                min_cost = nb["cost"]
                best_neighbor = nb

        for nb in neighbors:
            suffix = ""
            if best_neighbor and nb["state"] == best_neighbor["state"]:
                suffix = " Choose this neighbour."
            print(f"• {format_state(nb['state'])} cost is {nb['cost']}.{suffix}")

        print()

        # If no better neighbor is found, or we hit a local maximum
        if best_neighbor is None or best_neighbor["cost"] >= current_cost:
            print(f"Final State: {format_state(current_state)}")
            break

        # Move to the chosen neighbor
        current_state = best_neighbor["state"]
        current_cost = best_neighbor["cost"]

        # CRITICAL CHECK: Stop immediately if the chosen state achieves the goal (cost = 0)
        if current_cost == 0:
            print(f"Final State: {format_state(current_state)}")
            break

if __name__ == "__main__":
    initial_input = [int(x) for x in input().strip().split()]
    print()

    problem = FourQueensProblem(initial_input)
    hill_climbing(problem)

3 1 2 0 

Initial State: 3 1 2 0

Solution: Neighbours of 3 1 2 0 (cost is 2.)

• 1 3 2 0 cost is 1. Choose this neighbour.
• 2 1 3 0 cost is 1.
• 0 1 2 3 cost is 6.
• 3 2 1 0 cost is 6.
• 3 0 2 1 cost is 1.
• 3 1 0 2 cost is 1.

Neighbours of 1 3 2 0 (cost is 1.)

• 3 1 2 0 cost is 2.
• 2 3 1 0 cost is 2.
• 0 3 2 1 cost is 4.
• 1 2 3 0 cost is 4.
• 1 0 2 3 cost is 2.
• 1 3 0 2 cost is 0. Choose this neighbour.

Final State: 1 3 0 2
